# 6. Volatility Surface (EVC)

Surface sigma_evc par moneyness bucket, smoothed vs raw, comparaison avec la volatilité réalisée.

In [1]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("CLICKHOUSE_HOST", "localhost")
os.environ.setdefault("CLICKHOUSE_PORT", "8124")
os.environ.setdefault("CLICKHOUSE_DATABASE", "analytics")
os.environ.setdefault("CLICKHOUSE_USER", "default")
os.environ.setdefault("CLICKHOUSE_PASSWORD", "1234")

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

from core.db import ClickHouseDB, ClickHouseSettings

db = ClickHouseDB.from_settings(ClickHouseSettings())

POOL = "0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640"
START_DATE = "2026-04-01"
END_DATE   = "2026-04-14"

# Auto-cleanup: close CH connection on kernel shutdown so sockets don't pile up
import atexit as _atexit
_atexit.register(lambda: db.close() if 'db' in globals() and db is not None else None)


## 6.1 Vol Surface Data

In [2]:
vol_surface = db.read_query(f"""
    SELECT
        day, moneyness_bucket,
        sigma_evc_raw, sigma_evc_smoothed,
        tick_count, total_liquidity,
        rv_realized_7d, rv_realized_30d
    FROM {db.qualify('py_vol_surface_daily')}
    WHERE lower(pool) = lower('{POOL}')
      AND day >= toDate('{START_DATE}') AND day <= toDate('{END_DATE}')
    ORDER BY day, moneyness_bucket
""")
print(f"{vol_surface.height} rows, {vol_surface['day'].n_unique()} days, {vol_surface['moneyness_bucket'].n_unique()} buckets")
vol_surface.head(5)

9 rows, 6 days, 3 buckets


day,moneyness_bucket,sigma_evc_raw,sigma_evc_smoothed,tick_count,total_liquidity,rv_realized_7d,rv_realized_30d
date,f64,f64,f64,i64,f64,f64,f64
2026-04-01,-0.05,0.114944,0.107466,9,1.1888e20,0.558981,0.608865
2026-04-01,0.0,0.110387,0.099243,381,1.8534e22,0.558981,0.608865
2026-04-02,0.0,0.101016,0.100992,368,1.4924e22,0.507319,0.619862
2026-04-02,0.05,0.114419,0.105967,121,1.7836e21,0.507319,0.619862
2026-04-03,0.0,0.085932,0.097885,138,9.9694e21,0.437049,0.568163


## 6.2 Vol Smile (latest day)

In [3]:
latest = vol_surface.filter(pl.col("day") == vol_surface["day"].max())
buckets = latest["moneyness_bucket"].cast(pl.Float64).to_list()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=buckets, y=latest["sigma_evc_raw"].cast(pl.Float64).to_list(),
    mode="lines+markers", name="Raw",
))
fig.add_trace(go.Scatter(
    x=buckets, y=latest["sigma_evc_smoothed"].cast(pl.Float64).to_list(),
    mode="lines+markers", name="Smoothed",
))
# Add RV reference lines
rv7 = latest.filter(pl.col("rv_realized_7d").is_not_null())["rv_realized_7d"].cast(pl.Float64).mean()
rv30 = latest.filter(pl.col("rv_realized_30d").is_not_null())["rv_realized_30d"].cast(pl.Float64).mean()
if rv7 is not None:
    fig.add_hline(y=rv7, line_dash="dash", line_color="orange", annotation_text="RV 7d")
if rv30 is not None:
    fig.add_hline(y=rv30, line_dash="dot", line_color="red", annotation_text="RV 30d")

fig.add_vline(x=0, line_dash="dash", line_color="gray", annotation_text="ATM")
fig.update_layout(
    title=f"EVC Vol Smile — {latest['day'].max()}",
    xaxis_title="Log-Moneyness Bucket",
    yaxis_title="Sigma (annualized)",
    yaxis_tickformat=".0%",
    height=450,
)
fig.show()

## 6.3 Vol Surface Heatmap (day x moneyness)

In [4]:
pivot = vol_surface.pivot(
    on="day", index="moneyness_bucket", values="sigma_evc_smoothed"
).sort("moneyness_bucket")

bucket_labels = pivot["moneyness_bucket"].cast(pl.Float64).to_list()
date_cols = [c for c in pivot.columns if c != "moneyness_bucket"]
z = pivot.select(date_cols).fill_null(0).cast(pl.Float64).to_numpy()

fig = go.Figure(go.Heatmap(
    z=z, x=date_cols, y=bucket_labels,
    colorscale="Viridis", colorbar_title="Sigma",
))
fig.update_layout(
    title="EVC Vol Surface (smoothed)",
    xaxis_title="Date", yaxis_title="Log-Moneyness",
    height=500,
)
fig.show()

## 6.4 ATM Sigma EVC vs Realized Vol (time-series)

In [5]:
# ATM = bucket closest to 0
atm = vol_surface.filter(pl.col("moneyness_bucket").cast(pl.Float64).abs() < 0.05).sort("day")
atm_days = atm["day"].to_list()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=atm_days, y=atm["sigma_evc_smoothed"].cast(pl.Float64).to_list(),
    mode="lines+markers", name="Sigma EVC (ATM, smoothed)",
))
fig.add_trace(go.Scatter(
    x=atm_days, y=atm["rv_realized_7d"].cast(pl.Float64).to_list(),
    mode="lines", name="RV 7d",
))
fig.add_trace(go.Scatter(
    x=atm_days, y=atm["rv_realized_30d"].cast(pl.Float64).to_list(),
    mode="lines", name="RV 30d",
))
fig.update_layout(
    title="ATM Implied (EVC) vs Realized Vol",
    yaxis_title="Sigma (annualized)", yaxis_tickformat=".0%",
    height=400,
)
fig.show()

## 6.5 Implied Vol Snapshot — Tick-level Distribution

Sigma EVC par tick pour un snapshot donné (une heure), converti en strike price pour visualiser le smile à granularité fine.

In [6]:
import math

# Fetch the latest available hour as snapshot
snapshot = db.read_query(f"""
    SELECT
        hour,
        tick,
        toFloat64(sigma_evc_last) AS sigma_evc,
        toFloat64(tick_tvl_token0) AS tick_tvl,
        toFloat64(tick_liquidity_end) AS tick_liquidity
    FROM {db.qualify('mart_tick_sigma_evc_hourly')}
    WHERE lower(pool) = lower('{POOL}')
      AND sigma_evc_last IS NOT NULL
      AND tick_liquidity_end > 0
    ORDER BY hour DESC, tick
    LIMIT 10000
""")

# Take the most recent hour
latest_hour = snapshot["hour"].max()
snap = snapshot.filter(pl.col("hour") == latest_hour).sort("tick")
print(f"Snapshot: {latest_hour}, {snap.height} ticks")

# Get spot price and decimals for this hour
spot = db.read_query(f"""
    SELECT toFloat64(close_price) AS close_price
    FROM {db.qualify('int_daily_close_dense')}
    WHERE lower(pool) = lower('{POOL}')
    ORDER BY date DESC LIMIT 1
""")["close_price"][0]

decimals = db.read_query(f"""
    SELECT toInt32(token0_decimals) AS d0, toInt32(token1_decimals) AS d1
    FROM {db.qualify('stg_pools')}
    WHERE lower(pool) = lower('{POOL}')
""")
d0, d1 = int(decimals["d0"][0]), int(decimals["d1"][0])

# Convert tick to strike price (token0-per-token1, same as close_price)
snap = snap.with_columns(
    pl.col("tick").map_elements(
        lambda t: 1.0 / (math.exp(int(t) * math.log(1.0001)) * 10 ** (d0 - d1)),
        return_dtype=pl.Float64,
    ).alias("strike_price"),
)

print(f"Spot: ${spot:.2f}, Strike range: ${snap['strike_price'].min():.2f} — ${snap['strike_price'].max():.2f}")
snap.head(5)

Snapshot: 2026-04-06 11:00:00, 16 ticks
Spot: $2341.17, Strike range: $2148.49 — $2152.79


hour,tick,sigma_evc,tick_tvl,tick_liquidity,strike_price
datetime[μs],i64,f64,f64,f64,f64
2026-04-06 11:00:00,199575,0.035852,168408.227631,1.4521e19,2152.787783
2026-04-06 11:00:00,199576,0.099175,134723.228565,1.4521e19,2152.572526
2026-04-06 11:00:00,199579,0.192997,33678.335788,1.4521e19,2151.926883
2026-04-06 11:00:00,199583,0.034277,235639.863758,1.4519e19,2151.066327
2026-04-06 11:00:00,199584,0.052196,201971.987093,1.4519e19,2150.851242


In [7]:
# --- Chart 1: Vol Smile by Strike Price ---
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=[
        f"Implied Vol (sigma EVC) by Strike — {latest_hour}",
        "TVL Distribution by Strike (token0 equiv.)",
    ],
    row_heights=[0.6, 0.4],
)

strikes = snap["strike_price"].to_list()
sigmas = snap["sigma_evc"].to_list()
tvls = snap["tick_tvl"].to_list()

# Sigma scatter (color by TVL for liquidity weighting intuition)
fig.add_trace(go.Scatter(
    x=strikes, y=sigmas,
    mode="markers",
    marker=dict(
        size=6, color=tvls, colorscale="Blues", showscale=True,
        colorbar=dict(title="TVL ($)", x=1.02),
    ),
    name="sigma_evc",
    hovertemplate="Strike: $%{x:,.0f}<br>Sigma: %{y:.2%}<br>TVL: $%{marker.color:,.0f}<extra></extra>",
), row=1, col=1)

# Spot reference
fig.add_vline(x=spot, line_dash="dash", line_color="red", annotation_text=f"Spot ${spot:,.0f}", row=1, col=1)
fig.add_vline(x=spot, line_dash="dash", line_color="red", row=2, col=1)

# TVL bar chart
fig.add_trace(go.Bar(
    x=strikes, y=tvls,
    marker_color="rgba(55, 128, 191, 0.6)",
    name="TVL",
    hovertemplate="Strike: $%{x:,.0f}<br>TVL: $%{y:,.0f}<extra></extra>",
), row=2, col=1)

fig.update_yaxes(title_text="Sigma (annualized)", tickformat=".0%", row=1, col=1)
fig.update_yaxes(title_text="TVL ($)", row=2, col=1)
fig.update_xaxes(title_text="Strike Price (USDC/ETH)", row=2, col=1)
fig.update_layout(height=700, showlegend=False)
fig.show()

In [8]:
# --- Chart 2: Liquidity-weighted Vol Distribution (histogram) ---
# Weight sigma by TVL to show where the "market-implied" vol sits
snap_weighted = snap.filter(pl.col("tick_tvl") > 0)

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=snap_weighted["sigma_evc"].to_list(),
    nbinsx=50,
    name="Unweighted",
    opacity=0.5,
    marker_color="steelblue",
))

# Weighted histogram: repeat sigma values proportional to TVL
weights = snap_weighted["tick_tvl"].to_list()
total_w = sum(weights)
norm_weights = [w / total_w * len(weights) for w in weights]  # normalize to same count
weighted_sigmas = []
for s, w in zip(snap_weighted["sigma_evc"].to_list(), norm_weights):
    weighted_sigmas.extend([s] * max(1, int(w)))

fig.add_trace(go.Histogram(
    x=weighted_sigmas,
    nbinsx=50,
    name="TVL-weighted",
    opacity=0.5,
    marker_color="coral",
))

# Weighted median
snap_sorted = snap_weighted.sort("sigma_evc")
cum_tvl = snap_sorted["tick_tvl"].cum_sum()
half = cum_tvl[-1] / 2
weighted_median = float(snap_sorted.filter(cum_tvl >= half)["sigma_evc"][0])
fig.add_vline(x=weighted_median, line_dash="dash", line_color="red",
              annotation_text=f"TVL-weighted median: {weighted_median:.1%}")

fig.update_layout(
    title=f"Implied Vol Distribution across Ticks — {latest_hour}",
    xaxis_title="Sigma EVC (annualized)", xaxis_tickformat=".0%",
    yaxis_title="Count",
    barmode="overlay", height=400,
)
fig.show()

print(f"Ticks: {snap_weighted.height}, TVL-weighted median sigma: {weighted_median:.2%}")

Ticks: 16, TVL-weighted median sigma: 5.22%
